# 02 · Pré-processamento

Define e valida separações, transformações e remoções das features

In [6]:
from pathlib import Path
import joblib
import json

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


from regime_anomaly.config import (
    RANDOM_STATE, MODELS_DIR, PROCESSED_DIR, 
    METADATA_DIR, RAW_DIR,
    ensure_artifact_dirs
)

from regime_anomaly.data import LABEL_COLUMN, load_data

ensure_artifact_dirs()

In [ ]:
from regime_anomaly.preprocessing import (
    FEATURE_CATEGORICAL, FEATURE_HOUR, FEATURE_MONTH, FEATURE_NUMERIC,
    build_preprocessor, describe_preprocessor, split_columns,
)

In [ ]:
# Definição dos Diretórios
RAW = Path("../data/raw/Steel_industry_data.csv")
PROCESSED = Path(PROCESSED_DIR)
MODELS = Path(MODELS_DIR)
METADATA = Path(METADATA_DIR)

### 1) Importação da Fonte de Dados

In [5]:
df = load_data()
target_column = LABEL_COLUMN

df["date"] = pd.to_datetime(df["date"], dayfirst=True)
df = df.sort_values(by="date").reset_index(drop=True)

df.head()

,date,usage_kwh,lagging_current_reactive_power_kvarh,leading_current_reactive_power_kvarh,co2_tco2,lagging_current_power_factor,leading_current_power_factor,nsm,weekstatus,day_of_week,load_type
0,2018-01-01 00:00:00,3.42,3.46,0.0,0.0,70.30,100.0,0,Weekday,Monday,Light_Load
1,2018-01-01 00:15:00,3.17,2.95,0.0,0.0,73.21,100.0,900,Weekday,Monday,Light_Load
2,2018-01-01 00:30:00,4.00,4.46,0.0,0.0,66.77,100.0,1800,Weekday,Monday,Light_Load
3,2018-01-01 00:45:00,3.24,3.28,0.0,0.0,70.28,100.0,2700,Weekday,Monday,Light_Load
4,2018-01-01 01:00:00,3.31,3.56,0.0,0.0,68.09,100.0,3600,Weekday,Monday,Light_Load


### 2) Split Temporal

Os dados serão dividos em treino e teste por meio de um split temporal de 80 % de treino e 20% de teste 

In [34]:
# Definição do Eixo X e do Eixo Y
X = df.drop(columns=[target_column])
y = df[target_column].copy()

# Split temporal
cut = int(len(df) * 0.8)

X_train, X_test = X.iloc[:cut].copy(), X.iloc[cut:].copy()
y_train, y_test = y.iloc[:cut].copy(), y.iloc[cut:].copy()

print(f'Treino: {X_train['date'].min()} -> {X_train['date'].max()}')
print(f'Teste: {X_test['date'].min()} -> {X_test['date'].max()}')

Treino: 2018-01-01 00:00:00 -> 2018-10-19 23:45:00
Teste: 2018-10-20 00:00:00 -> 2018-12-31 23:45:00


### 3) Adaptação das Features

O Processo adapta as features numéricas para alinhar melhor a distância
entre os pontos, e evitar redundância

In [35]:
def time_feature_adapt(data: pd.DataFrame) -> pd.DataFrame:
    """
    Será realizado a adaptação das colunas relacionadas a tempo, 
    de modo a alinhar melhor a distância entre os pontos
    (não será adpatado a data para dia da semana por 
    apresentar redundância com a coluna 'Day_of_Week')
    """
    out = data.copy()
    
    out['hour_sin'] = np.sin(2 * np.pi * out["date"].dt.hour / 24)
    out['hour_cos'] = np.cos(2 * np.pi * out["date"].dt.hour / 24)
    out["month_sin"] = np.sin(2 * np.pi * (out["date"].dt.month - 1) / 12)
    out["month_cos"] = np.cos(2 * np.pi * (out["date"].dt.month - 1) / 12)
    #out['NSM_sin'] = np.sin(2 * np.pi * out['NSM'] / 86400)
    #out['NSM_cos'] = np.cos(2 * np.pi * out['NSM'] / 86400)
    
    return out.drop(columns=["date", "NSM"])

X_train_featured = time_feature_adapt(X_train)
X_test_featured = time_feature_adapt(X_test)

In [36]:
FEATURE_NUMERIC = [
    'Usage_kWh', # feature central
    'Lagging_Current_Reactive.Power_kVarh',  # r=0.9 com o 'Usage_kWh'
    'Leading_Current_Reactive_Power_kVarh',  # r=-0.32 com o 'Usage_kWh'
    'Lagging_Current_Power_Factor'           # r=0.39 com o 'Usage_kWh'
]

FEATURE_HOUR = ['hour_sin', 'hour_cos']
FEATURE_MONTH = ['month_sin', 'month_cos']
FEATURE_CLUSTERS_CATEGORICAL = ['Day_of_week']

FEATURE_REPORTED = [
    "CO2(tCO2)", # r=0.99 com o 'Usage_kWh' -> excluída no K-Means e mantida para validar e reportar os clusters
    "WeekStatus" # redundante com o Day_of_week
]

FEATURE_DELETED = [
    "date",  # substituída por hour_sin/cos e month_sin/cos
    "NSM",   # substituída por hour_sin/cos (mesma informação, menos granular)
]

### 4) Pré-Processamento

In [ ]:
def create_preprocessing(df_features, cols):
    """ Transformação do subconjunto em escala numérica e One-Hot Encoder """

    numeric_cols = [x for x in cols if x in FEATURE_NUMERIC]
    categorical_cols = [x for x in cols if x in FEATURE_CLUSTERS_CATEGORICAL]

    column_transfer_preprocessor = ColumnTransformer(
        transformers=[
            ("num", StandardScaler(), numeric_cols),
            ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_cols)
        ]
    )

    return column_transfer_preprocessor.fit_transform(df_features)


def evaluate_config(cols, df_features, y, use_pca):
    """ 
    Além de Pre-Processar os dados, treina o K-Means e
    avalia os resultados por meio das métricas Silhouette e ARI
    """
    X_prep = create_preprocessing(df_features=df_features, cols=cols)

    if use_pca:
        pca = PCA(n_components=0.90, random_state=42)
        X_prep = pca.fit_transform(X_prep)
    
    km = KMeans(n_clusters=3, random_state=42, n_init=10).fit(X_prep)
    silhouette_scalled = silhouette_score(X_prep, km.labels_)
    ari_score = adjusted_rand_score(y, km.labels_)

    return silhouette_scalled, ari_score


def stability_subsample(features, n_clusters=3, n_runs=30, sample_frac=0.8, seed=42):
    """
    Reamostra 80% dos dados, refita o K-Means e mede o ARI
    na interseção de pontos com a solução de referência.
    """
    random_rng = np.random.default_rng(seed=seed)
    km_reference = KMeans(n_clusters=n_clusters, random_state=seed, n_init=10).fit(features)
    ref_labels = km_reference.labels_
    n = len(features)
    size = int(n * sample_frac)

    list_scores = []
    for _ in range(n_runs):
        index = random_rng.choice(n, size=size, replace=False)
        km = KMeans(n_clusters=n_clusters, random_state=seed, n_init=10).fit(features[index])
        list_scores.append( adjusted_rand_score(ref_labels[index], km.labels_) )

    return float(np.mean(list_scores)), float(np.std(list_scores))


#### 4.1) Diagnóstico da Perfórmance de Dados

- o **Coeficiente de Silhueta** avalia o quão bem cada ponto está agrupado em seu cluster
  em comparação com os clusters vizinhos (métrica de -1 a 1)

    * maior que 0.5: Bem preparados
    * entre 0.25 e 0.5: Grupos razoáveis, mas que devem ser usados com cautela
    * entre 0.1 e 0.24: Grupos Existentes, mas não são natos
    * menor que 0.1: Sem forte interpretação

- o **ARI** mede a semelhança entre 2 grupos diferentes, variando de -1 a 1
    * 1: agrupamentos idênticos
    * 0: valores aleatórios
    * -1: discordância maior que o acaso


In [ ]:
configs = {
    "Base (Todas as Features Numéricas)": FEATURE_NUMERIC,
    "Base + hora": FEATURE_NUMERIC + FEATURE_HOUR,
    "Base + mês": FEATURE_NUMERIC + FEATURE_MONTH,
    "Base + hora + mês": FEATURE_NUMERIC + FEATURE_HOUR + FEATURE_MONTH,
    "Base + Day_of_Week": FEATURE_NUMERIC + ['Day_of_Week'],
    "Base + Day_of_Week + PCA90": FEATURE_NUMERIC + ['Day_of_Week']
}

results = []

for name, cols in configs.items():
    use_pca = False if name != "Base + Day_of_Week + PCA90" else True
    sil, ari = evaluate_config(cols, X_train_featured, y_train, use_pca=use_pca)
    print(f'Name: {name} | Silhouette: {sil}| ARI: {ari}')


Name: Base (Todas as Features Numéricas) | Silhouette: 0.6361124912454764| ARI: 0.3481584240769701
Name: Base + hora | Silhouette: 0.6361124912454764| ARI: 0.3481584240769701
Name: Base + mês | Silhouette: 0.6361124912454764| ARI: 0.3481584240769701
Name: Base + hora + mês | Silhouette: 0.6361124912454764| ARI: 0.3481584240769701
Name: Base + Day_of_Week | Silhouette: 0.6361124912454764| ARI: 0.3481584240769701
Name: Base + Day_of_Week + PCA90 | Silhouette: 0.6981876636382569| ARI: 0.34782484622824295


#### 4.2) Início do Pré-Processamento

In [25]:
preprocessor = ColumnTransformer(
        transformers=[
            ("num", StandardScaler(), FEATURE_NUMERIC),
            ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), FEATURE_CLUSTERS_CATEGORICAL)
        ]
    )

pipeline = Pipeline(steps=[
    ("prep", preprocessor),
    ("pca",  PCA(n_components=0.90, random_state=42)),
])

# Escalado + PCA
X_train_processed = pipeline.fit_transform(X_train_featured)
X_test_processed = pipeline.transform(X_test_featured) # é utilizado somente 'transform' pata evitar vazamento de dados do teste para o treino

components = pipeline.named_steps["pca"].n_components_

print(f"O Pipeline manteve {components} componetes para explicar 90% da variância")

O Pipeline manteve 6 componetes para explicar 90% da variância


#### 4.3) Checagens

avaliar se as tabelas pré-processadas possuem alguma inconsistência (estrutura diferente, tamanhos diferentes, não apresentam valor)

In [26]:
# Apresenta a mesma forma
assert X_train_processed.shape[1] == X_test_processed.shape[1], \
    "Treino e teste têm números de componentes diferentes"

# Não apresenta valores nulos ou infinitos
assert not np.isnan(X_train_processed).any(), "Há NaN no X_train"
assert not np.isnan(X_test_processed).any(),  "Há NaN no X_test"
assert not np.isinf(X_train_processed).any(), "Há inf no X_train"
assert not np.isinf(X_test_processed).any(),  "Há inf no X_test"

# Split não perdeu linhas
assert len(X_train) + len(X_test) == len(df), "Split perdeu ou duplicou linhas"

#### 4.4) Correlações

Dos Clusters apresentados, será availado o quanto cada um se correlaciona entre si,
teria que ser uma correlação 0 para destacar que cada um agrupa classificações diferentes

In [27]:
comp_df = pd.DataFrame(X_train_processed, columns=[f"PC{i+1}" for i in range(components)])
corr_pos = comp_df.corr().abs()

upper = corr_pos.where(np.triu(np.ones(corr_pos.shape), k=1).astype(bool))
print(f"Correlação máxima entre componentes (deve ser ~0): {upper.stack().max().round(6)}")

Correlação máxima entre componentes (deve ser ~0): 0.0


#### 4.5) Validação Externa de Clusters

Verificar se os clusters do pipeline se alinham com as features que não foram incluídas nos clusters
  
   * 'CO2(tCO2)' -> redundancia com a coluna 'Usage_kWh'
   * 'Leading_Current_Power_Factor' -> suspeita de constância

In [28]:
# Treinamento com as features selecionadas
km_final = KMeans(n_clusters=3, random_state=42, n_init=10).fit(X_train_processed)
labels_sel = km_final.labels_

In [29]:
# Coloca as labels novamente com as features antes removidas
df_check = X_train.copy()
df_check['cluster'] = labels_sel

# Calcula a variabilidade das features removidas por cluster
for feature in ['CO2(tCO2)', 'Leading_Current_Power_Factor']:

    mean_per_cluster = df_check.groupby("cluster")[feature].mean()
    variance = mean_per_cluster.max() - mean_per_cluster.min()
    
    std_global = df_check[feature].std()
    ratio = variance/std_global if std_global > 0 else 0

    print(f"{feature}:")
    print(f"- médias por cluster: {mean_per_cluster.round(3).to_dict()}")
    print(f"- variação entre clusters: {variance:.4f}")
    print(f"- std entre clusters: {std_global:.4f}")
    print(f"- razão variação/std global: {ratio:.3f}\n")


CO2(tCO2):
- médias por cluster: {0: 0.031, 1: 0.0, 2: 0.003}
- variação entre clusters: 0.0309
- std entre clusters: 0.0164
- razão variação/std global: 1.891

Leading_Current_Power_Factor:
- médias por cluster: {0: 99.991, 1: 99.96, 2: 38.716}
- variação entre clusters: 61.2752
- std entre clusters: 30.3782
- razão variação/std global: 2.017



#### 4.6) Estabilidade

Avalia como o K-Means se comporta diante de diferentes sementes e sub-amostras
e se os clusters são os mesmos, caso sejam são estáveis

    - acima de 0.8: Estável
    - entre 0.6 e 0.8: Moderavelmente Estável
    - abaixo de 0.6: Instável

In [30]:
sil = silhouette_score(X=X_train_processed, labels=km_final.labels_)
mean_sub, std_sub = stability_subsample(X_train_processed)

print(f'Silhouette: {sil}')
print(f'ARI - Estabilidade: {mean_sub:.4f} +- {std_sub:.4f}')

Silhouette: 0.5409003505961558
ARI - Estabilidade: 0.9983 +- 0.0015


### 5) Salvamento de Dados

In [31]:
# Processamento de dados (treino e teste)
np.save(PROCESSED / "x_train.npy", X_train_processed)
np.save(PROCESSED / "x_test.npy", X_test_processed)

y_train.to_csv(PROCESSED / "y_train.csv", index=False)
y_test.to_csv(PROCESSED / "y_test.csv", index=False)

In [32]:
# DataFrames Enriquecidos
train_enriched = X_train.copy()
test_enriched = X_test.copy()

train_enriched['hour'] = train_enriched["date"].dt.hour
train_enriched['hour_sin'] = np.sin(2 * np.pi * train_enriched['hour'] / 24)
train_enriched['hour_cos'] = np.cos(2 * np.pi * train_enriched['hour'] / 24)

train_enriched['month'] = train_enriched["date"].dt.month
train_enriched['month_sin'] = np.sin(2 * np.pi * (train_enriched['month'] - 1) / 12)
train_enriched['month_cos'] = np.cos(2 * np.pi * (train_enriched['month'] - 1) / 12)


for i in range(X_train_processed.shape[1]):
    train_enriched[f"PC{i+1}"] = X_train_processed[:, i]
    test_enriched[f"PC{i+1}"] = X_test_processed[:, i]


train_enriched.to_csv(PROCESSED / "X_train_enriched.csv", index=False)
test_enriched.to_csv(PROCESSED / "X_test_enriched.csv", index=False)

# Pipeline Treinado
joblib.dump(pipeline, MODELS / "preprocessor.joblib")

['..\\models\\preprocessor.joblib']

In [ ]:
# Definição das Features
feature_data = {
    "raw_file": str(RAW),
    "chosen_config": "numeric + day_of_week + pca90",

    "feature_numeric": FEATURE_NUMERIC,
    "feature_categoric": FEATURE_CLUSTERS_CATEGORICAL,
    "feature_reporting_only": FEATURE_REPORTED,
    "feature_deleted": FEATURE_DELETED,
    "target": target_column,

    "pca_components": [f"PC{i+1}" for i in range(components)],
    "one_hot_encoding_features": list(
            pipeline.named_steps["prep"]\
            .named_transformers_["cat"]\
            .get_feature_names_out(FEATURE_CLUSTERS_CATEGORICAL)
    ),

    "split_strategy": "temporal_80_20",
    "n_rows_total": int(len(df)),
    "n_train": int(len(X_train)),
    "n_test": int(len(X_test)),
    "train_date_range": [str(X_train["date"].min()), str(X_train["date"].max())],
    "test_date_range":  [str(X_test["date"].min()),  str(X_test["date"].max())],
}

(METADATA / "metadata.json").write_text(
    json.dumps(feature_data, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

996